# 🎙️ 個人聲音克隆與文字朗讀 (Voice Clone TTS) - Colab GPU 執行手冊

本筆記本可直接於 **Google Colab (啟用免費 GPU)** 運行，並可隨時將程式碼推播至您的 GitHub Repository。

### ⚠️ 執行前第一步：確認 GPU 加速器
請點擊 Colab 上方選單：`執行階段 (Runtime)` -> `變更執行階段類型 (Change runtime type)` -> 選取 **T4 GPU** -> 儲存。

In [1]:
# 步驟 1: 驗證 GPU 環境
!nvidia-smi

Mon Oct 19 06:26:30 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# 步驟 2: 安裝高品質聲音克隆核心與 Web 工具 (以 F5-TTS 為例)
!pip install -q f5-tts gradio torchaudio soundfile pyngrok

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.6/107.6 kB 6.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 60.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 73.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.8/168.8 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 840.2/840.2 kB 60.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 118.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 283.6/283.6 kB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [4]:
# 步驟 3: 建立聲音克隆與推論程式
import os
import torchaudio
from f5_tts.api import F5TTS

# 最新版 API：直接初始化即可（預設即為 F5-TTS 模型）
f5tts_engine = F5TTS()
print("✅ 聲音克隆模型載入成功！")

Download Vocos from huggingface charactr/vocos-mel-24khz


config.yaml:   0%|          | 0.00/461 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 54.4MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

F5TTS_v1_Base/model_1250000.safetensors: reconstructing file:   0%|          |  0.00B / 1.35GB            

F5TTS_v1_Base/model_1250000.safetensors: downloading bytes:           |  0.00B            


vocab :  /usr/local/lib/python3.13/dist-packages/f5_tts/infer/examples/vocab.txt
token :  custom
model :  /root/.cache/huggingface/hub/models--SWivid--F5-TTS/snapshots/84e5a410d9cead4de2f847e7c9369a6440bdfaca/F5TTS_v1_Base/model_1250000.safetensors 

✅ 聲音克隆模型載入成功！


In [11]:
# 步驟 4: 長文字自動切句與無縫串接推論函式
import re
import soundfile as sf
import numpy as np

def split_text_into_chunks(text, max_chars=35):
    """
    依據標點符號將長文章拆解為 20~35 字的合理短句
    """
    # 以標點符號進行切割
    sentences = re.split(r'([，。！？；\n,.!?;])', text)
    chunks = []
    current = ""

    for part in sentences:
        if not part:
            continue
        current += part
        # 遇到標點或累積字數達到上限就獨立成一句
        if part in "，。！？；\n,.!?;" and len(current) >= 15:
            chunks.append(current.strip())
            current = ""
        elif len(current) >= max_chars:
            chunks.append(current.strip())
            current = ""

    if current.strip():
        chunks.append(current.strip())

    return [c for c in chunks if c]

def clone_long_voice(ref_audio_path, ref_transcript, target_text, output_path="output_long.wav", progress=None):
    chunks = split_text_into_chunks(target_text)
    total = len(chunks)
    audio_segments = []
    sample_rate = 24000

    print(f"📄 長文已拆解為 {total} 個句子，開始分段生成...")

    # 建立 0.25 秒的自然換氣微小靜音 (Silence)
    silence = np.zeros(int(sample_rate * 0.25), dtype=np.float32)

    for i, chunk in enumerate(chunks):
        if progress:
            progress((i + 1) / total, desc=f"正在生成第 {i+1}/{total} 句: {chunk[:10]}...")

        # 呼叫 F5-TTS 推論單句
        res = f5tts_engine.infer(
            ref_file=ref_audio_path,
            ref_text=ref_transcript,
            gen_text=chunk
        )

        if isinstance(res, tuple):
            wav, sr = res[0], res[1]
        else:
            wav, sr = res, sample_rate

        sample_rate = sr
        audio_segments.append(wav)
        audio_segments.append(silence) # 句與句之間加入自然停頓

    # 將所有音訊陣列無縫串接
    final_audio = np.concatenate(audio_segments)
    sf.write(output_path, final_audio, sample_rate)
    return output_path

print("✅ 長文串接模組已載入完成！")

✅ 長文串接模組已載入完成！


In [12]:
# 步驟 5: 啟動支援長影片文案的 Web 介面
import gradio as gr
import traceback

def tts_long_interface(ref_audio, ref_text, target_text, progress=gr.Progress()):
    if not ref_audio:
        return None, "⚠️ 請先錄製或上傳 3~8 秒的聲音樣本！"
    if not target_text.strip():
        return None, "⚠️ 請輸入想要朗讀的長文案內容！"

    try:
        out_file = clone_long_voice(
            ref_audio_path=ref_audio,
            ref_transcript=ref_text.strip() if ref_text else "",
            target_text=target_text.strip(),
            progress=progress
        )
        return out_file, "🎉 整段長音訊合成完成！可直接點播放器右側三個點下載成音檔。"
    except Exception as e:
        err_msg = traceback.format_exc()
        print("詳細錯誤資訊:\n", err_msg)
        return None, f"❌ 合成失敗，原因：{str(e)}"

ui = gr.Interface(
    fn=tts_long_interface,
    inputs=[
        gr.Audio(type="filepath", label="1. 錄製或上傳 3~8 秒聲音樣本 (維持 5 秒效果最好)"),
        gr.Textbox(label="2. 樣本語音逐字稿 (務必與錄音一致)", value="今天你畫線了嗎？"),
        gr.Textbox(
            label="3. 想要朗讀的完整腳本 (支援一分多鐘長文，約 200~300 字)",
            lines=10,
            placeholder="請在此貼上一整篇短影音腳本文案，系統會自動切句並依序朗讀合成..."
        ),
    ],
    outputs=[
        gr.Audio(label="合成的一分鐘完整音檔 (可直接下載 MP3/WAV)"),
        gr.Textbox(label="處理進度與狀態")
    ],
    title="🎙️ 個人聲音克隆 - 長篇短影音旁白生成器",
    description="💡 提示：短影音 60 秒大約需要 220 ~ 260 字。貼上完整文章後點擊 Submit，系統會依進度條逐句渲染成一整段完整音檔。"
)

ui.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://9b35fff86ff7fadc56.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
